# Stage 2 v4: BN gamma importance proxy

## v3的问题回顾

- v3保守版: EKF 90.07% vs Random 92.37%, **EKF反向落后 -2.30%**
- 根因: `activation magnitude` 在BN架构下不反映channel真实重要性
- BN的γ参数才是structured pruning的标准信号 (Network Slimming, ICCV 2017)

## v4改动

把EKF的观测 `z_{i,k}` 从 `activation.abs().mean()` 换成 `|γ_i|` (BN第二层的scale参数).

其他机制全部保留: warm-up, 相对quantile阈值, sigmoid soft gate, std归一化.

预期: EKF应该明显跑赢Random (BN γ本来就是channel重要性的标准信号).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/ekf_adaptive_pruning')
print('[cwd]', os.getcwd())

## 覆盖 ekf_pruner.py

关键改动: `EKFChannelGate` 现在接受一个 `obs_fn` 参数，可以外部传入观测计算方式。
保留 activation 版本 (v2/v3) 用于对比，新增 BN γ 版本 (v4)。

In [ ]:
ekf_code = '''"""EKF-based channel importance tracking and soft gating (v4: BN gamma proxy)."""
import torch
import torch.nn as nn
import torch.nn.functional as F


def softmax_entropy(logits):
    p = F.softmax(logits, dim=-1)
    logp = F.log_softmax(logits, dim=-1)
    return -(p * logp).sum(dim=-1)


def max_softmax_prob(logits):
    p = F.softmax(logits, dim=-1)
    return 1.0 - p.max(dim=-1).values


class ChannelEKF:
    """Per-channel independent scalar EKF."""

    def __init__(self, num_channels, Q=1e-3, R=1e-2, device="cuda"):
        self.num_channels = num_channels
        self.Q = Q
        self.R = R
        self.device = device
        self.theta = None
        self.P = torch.full((num_channels,), 1.0, device=device)

    def update(self, obs):
        if self.theta is None:
            self.theta = obs.clone()
            return
        P_pred = self.P + self.Q
        K = P_pred / (P_pred + self.R)
        self.theta = self.theta + K * (obs - self.theta)
        self.P = (1 - K) * P_pred


class EKFChannelGate(nn.Module):
    """Channel-wise soft gating with warm-up and relative threshold.

    obs_source: \"activation\" or \"bn_gamma\"
      - activation: z_i = mean(|x|) over batch/spatial dims (v2/v3)
      - bn_gamma:   z_i = |gamma_i| from an externally bound BN layer (v4)
    """

    def __init__(self, num_channels, obs_source="bn_gamma", bn_layer=None,
                 Q=1e-3, R=1e-2, alpha=3.0,
                 prune_quantile_min=0.1, prune_quantile_max=0.3,
                 warmup_steps=5, device="cuda"):
        super().__init__()
        self.num_channels = num_channels
        self.obs_source = obs_source
        self.bn_layer = bn_layer     # reference to the BN layer for gamma
        self.alpha = alpha
        self.prune_quantile_min = prune_quantile_min
        self.prune_quantile_max = prune_quantile_max
        self.warmup_steps = warmup_steps
        self.step = 0
        self.ekf = ChannelEKF(num_channels, Q=Q, R=R, device=device)
        self.current_uncertainty = 0.5
        self.last_gate = None
        self.last_keep_ratio = None

    def set_uncertainty(self, u):
        self.current_uncertainty = float(u)

    def compute_observation(self, x):
        """Return observation z_i per channel, shape [C]."""
        if self.obs_source == "activation":
            return x.abs().mean(dim=[0, 2, 3])
        elif self.obs_source == "bn_gamma":
            assert self.bn_layer is not None, "bn_layer must be bound for bn_gamma source"
            return self.bn_layer.weight.detach().abs()
        else:
            raise ValueError(f"Unknown obs_source: {self.obs_source}")

    def forward(self, x):
        B, C, H, W = x.shape
        with torch.no_grad():
            obs = self.compute_observation(x)
        self.ekf.update(obs)
        self.step += 1

        if self.step <= self.warmup_steps:
            self.last_gate = torch.ones(C, device=x.device)
            self.last_keep_ratio = 1.0
            return x

        q = self.prune_quantile_min + (self.prune_quantile_max - self.prune_quantile_min) * (1 - self.current_uncertainty)
        q = max(0.0, min(0.9, q))
        tau = torch.quantile(self.ekf.theta, q).item()

        gate = torch.sigmoid(self.alpha * (self.ekf.theta - tau) / (self.ekf.theta.std() + 1e-6))
        gate_bcast = gate.view(1, C, 1, 1)
        out = x * gate_bcast
        self.last_gate = gate.detach()
        self.last_keep_ratio = (gate > 0.5).float().mean().item()
        return out


class RandomChannelGate(nn.Module):
    def __init__(self, num_channels, keep_ratio=0.5, device="cuda"):
        super().__init__()
        self.num_channels = num_channels
        self.keep_ratio = keep_ratio
        self.last_keep_ratio = None

    def forward(self, x):
        B, C, H, W = x.shape
        n_keep = max(1, int(self.keep_ratio * C))
        perm = torch.randperm(C, device=x.device)
        mask = torch.zeros(C, device=x.device)
        mask[perm[:n_keep]] = 1.0
        self.last_keep_ratio = mask.mean().item()
        return x * mask.view(1, C, 1, 1)
'''

with open('models/ekf_pruner.py', 'w') as f:
    f.write(ekf_code)
print('ekf_pruner.py v4 写入完成:', os.path.getsize('models/ekf_pruner.py'), '字节')

## 覆盖 inference.py

关键改动: 安装gate时把BasicBlock内的bn2 layer绑定给EKFChannelGate.

In [ ]:
inference_code = '''"""Inference with BN gamma EKF gating (v4)."""
import os
import sys
import argparse

sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T

from models.resnet18_cifar import resnet18_cifar
from models.ekf_pruner import EKFChannelGate, RandomChannelGate, softmax_entropy


def get_test_loader(data_dir="./data", batch_size=128, num_workers=2):
    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2470, 0.2435, 0.2616)
    test_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
    test_set = torchvision.datasets.CIFAR10(root=data_dir, train=False, download=True, transform=test_tf)
    return DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)


def wrap_layer_with_ekf_gate(layer_seq, gate_kwargs, device):
    """Insert EKFChannelGate after each BasicBlock, binding bn2 as gamma source."""
    new_modules = []
    gate_list = []
    for block in layer_seq:
        new_modules.append(block)
        out_ch = block.conv2.out_channels
        # Bind the bn2 layer for BN gamma observation
        gate = EKFChannelGate(
            num_channels=out_ch,
            bn_layer=block.bn2,
            device=device,
            **gate_kwargs
        ).to(device)
        new_modules.append(gate)
        gate_list.append(gate)
    return nn.Sequential(*new_modules), gate_list


def wrap_layer_with_random_gate(layer_seq, keep_ratio, device):
    new_modules = []
    gate_list = []
    for block in layer_seq:
        new_modules.append(block)
        out_ch = block.conv2.out_channels
        gate = RandomChannelGate(num_channels=out_ch, keep_ratio=keep_ratio, device=device).to(device)
        new_modules.append(gate)
        gate_list.append(gate)
    return nn.Sequential(*new_modules), gate_list


def install_ekf_gates(model, gate_kwargs, target_layers=("layer3", "layer4"), device="cuda"):
    all_gates = []
    for lname in target_layers:
        layer = getattr(model, lname)
        new_seq, gates = wrap_layer_with_ekf_gate(layer, gate_kwargs, device)
        setattr(model, lname, new_seq)
        all_gates.extend(gates)
    return all_gates


def install_random_gates(model, keep_ratio, target_layers=("layer3", "layer4"), device="cuda"):
    all_gates = []
    for lname in target_layers:
        layer = getattr(model, lname)
        new_seq, gates = wrap_layer_with_random_gate(layer, keep_ratio, device)
        setattr(model, lname, new_seq)
        all_gates.extend(gates)
    return all_gates


def evaluate_nogate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100.0 * correct / total


def evaluate_with_ekf_gates(model, gates, loader, device, entropy_norm=2.3):
    model.eval()
    correct, total = 0, 0
    keep_ratios = [[] for _ in gates]
    running_unc = 0.5
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            for g in gates:
                g.set_uncertainty(running_unc)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
            ent = softmax_entropy(logits).mean().item()
            running_unc = min(1.0, ent / entropy_norm)
            for i, g in enumerate(gates):
                if g.last_keep_ratio is not None:
                    keep_ratios[i].append(g.last_keep_ratio)
    acc = 100.0 * correct / total
    avg_keep = [sum(r) / len(r) if r else 0.0 for r in keep_ratios]
    return acc, avg_keep


def evaluate_with_random_gates(model, gates, loader, device):
    model.eval()
    correct, total = 0, 0
    keep_ratios = [[] for _ in gates]
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
            for i, g in enumerate(gates):
                if g.last_keep_ratio is not None:
                    keep_ratios[i].append(g.last_keep_ratio)
    acc = 100.0 * correct / total
    avg_keep = [sum(r) / len(r) if r else 0.0 for r in keep_ratios]
    return acc, avg_keep


def load_base_model(ckpt_path, device):
    model = resnet18_cifar(num_classes=10).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    return model


def run_comparison(ckpt_path, device="cuda"):
    loader = get_test_loader()

    print("\\n=== Baseline (no pruning) ===")
    m = load_base_model(ckpt_path, device)
    acc_base = evaluate_nogate(m, loader, device)
    print(f"acc = {acc_base:.2f}%")

    print("\\n=== EKF adaptive gating (BN gamma) ===")
    m = load_base_model(ckpt_path, device)
    ekf_kwargs = dict(
        obs_source="bn_gamma",
        Q=1e-3, R=1e-2, alpha=3.0,
        prune_quantile_min=0.1, prune_quantile_max=0.3,
        warmup_steps=5,
    )
    gates = install_ekf_gates(m, ekf_kwargs, target_layers=("layer3", "layer4"), device=device)
    acc_ekf, keep_ekf = evaluate_with_ekf_gates(m, gates, loader, device)
    avg_keep_ekf = sum(keep_ekf) / len(keep_ekf)
    print(f"acc = {acc_ekf:.2f}%,  avg_keep_ratio = {avg_keep_ekf:.3f}")
    print(f"per-gate keep: {[round(r, 3) for r in keep_ekf]}")

    match_keep = max(0.1, min(0.95, avg_keep_ekf))

    print("\\n=== Random gating (matched keep ratio) ===")
    m = load_base_model(ckpt_path, device)
    gates = install_random_gates(m, keep_ratio=match_keep, target_layers=("layer3", "layer4"), device=device)
    acc_rand, keep_rand = evaluate_with_random_gates(m, gates, loader, device)
    avg_keep_rand = sum(keep_rand) / len(keep_rand)
    print(f"acc = {acc_rand:.2f}%,  avg_keep_ratio = {avg_keep_rand:.3f}")

    print("\\n=== Summary ===")
    print(f"  No pruning:      {acc_base:.2f}%")
    print(f"  EKF (BN gamma):  {acc_ekf:.2f}%  (keep {avg_keep_ekf:.2%})")
    print(f"  Random (match):  {acc_rand:.2f}%  (keep {avg_keep_rand:.2%})")
    print(f"  EKF advantage:   {acc_ekf - acc_rand:+.2f}% over random")
    print(f"  Accuracy drop:   {acc_base - acc_ekf:.2f}% vs no-prune baseline")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--ckpt", type=str, default="./checkpoints/resnet18_cifar_base.pth")
    args = parser.parse_args()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[device] {device}")
    run_comparison(args.ckpt, device=device)
'''

with open('src/inference.py', 'w') as f:
    f.write(inference_code)
print('inference.py v4 写入完成:', os.path.getsize('src/inference.py'), '字节')

## Smoke test: 确认BN γ真的被读到了

In [ ]:
import sys
sys.path.insert(0, '.')
for mod in list(sys.modules.keys()):
    if mod.startswith('models') or mod.startswith('src'):
        del sys.modules[mod]

import torch
from models.resnet18_cifar import resnet18_cifar
from models.ekf_pruner import EKFChannelGate

# Load trained model
model = resnet18_cifar(num_classes=10).cuda()
ckpt = torch.load('./checkpoints/resnet18_cifar_base.pth', map_location='cuda')
model.load_state_dict(ckpt['model_state_dict'])

# Pick layer3[0] block, look at its bn2 gamma distribution
block = model.layer3[0]
gamma = block.bn2.weight.detach().abs()
print(f'layer3[0].bn2 gamma shape: {gamma.shape}')
print(f'|gamma| min={gamma.min().item():.4f}, max={gamma.max().item():.4f}')
print(f'|gamma| mean={gamma.mean().item():.4f}, std={gamma.std().item():.4f}')
print(f'|gamma| quantile 0.1: {torch.quantile(gamma, 0.1).item():.4f}')
print(f'|gamma| quantile 0.3: {torch.quantile(gamma, 0.3).item():.4f}')
print()
print('如果 |gamma| 分布有明显 spread (max/min比值 > 3), 说明BN γ作为重要性信号有意义.')
print(f'max/min ratio: {(gamma.max() / (gamma.min() + 1e-8)).item():.2f}')

## 跑v4对比

In [ ]:
!python src/inference.py

## 预期结果

```
No pruning:      94.97%
EKF (BN gamma):  93-94%  (keep ~82%)
Random (match):  90-92%  (keep ~82%)
EKF advantage:   +2-3%
```

**关键判据**:
- EKF > Random **显著**正向领先 → 机制work, 可以继续深挖
- EKF ≈ Random → BN γ也没抓到重点, 要考虑更激进的方案
- EKF < Random → 可能有bug, 需要debug (但概率很小, BN γ是已验证的重要性信号)

如果v4跑通, 下一步可以做:
1. Trade-off曲线 (不同 quantile 下的accuracy vs keep ratio)
2. 消融实验 (去掉EKF只用瞬时BN γ, 对比收益)
3. 扩展到PathMNIST + ResNet50 (Stage 2)